<a href="https://colab.research.google.com/github/0v3rL4nd/oop-python/blob/main/Anatomia_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Giorno 1 (Avanzato): Anatomia di BERT ed Estrazione di Embeddings
Benvenuto! Questo notebook esplora BERT scendendo a basso livello: dai tensori del tokenizer, all'estrazione degli embedding (`[CLS]`), fino all'integrazione con modelli classici di machine learning.



```
# Questo è formattato come codice
```

# **I Transformer: Il "Motore" dell'IA Moderna**
Il Transformer non è un modello specifico, ma un'architettura di rete neurale inventata dai ricercatori di Google nel 2017 (nel celebre articolo scientifico "Attention Is All You Need").

Prima del 2017, i modelli linguistici (chiamati RNN o LSTM) leggevano il testo come facciamo noi esseri umani: una parola alla volta, da sinistra verso destra. Questo era un problema perché era un processo lentissimo e, arrivati alla fine di una frase lunga, il modello "dimenticava" le prime parole.

I Transformer hanno rivoluzionato il mondo dell'IA introducendo due concetti:

 - Lettura Parallela: Il Transformer non legge una parola alla volta, ma ingerisce l'intera frase tutta in una volta. Questo rende l'addestramento enormemente più veloce sfruttando la potenza delle schede video (GPU).

 - Meccanismo di Self-Attention (Auto-Attenzione): È il vero segreto. Quando il modello analizza una parola, calcola matematicamente quanto ogni altra parola della frase sia importante per darle un contesto.

Esempio: Nella frase "Ho mangiato una pesca" e "Sono andato a pesca", il meccanismo di attenzione fa sì che la parola "pesca" guardi verso "mangiato" nel primo caso, e verso "andato" nel secondo, capendo immediatamente che si tratta di due concetti completamente diversi.

I modelli di cui parliamo (BERT, GPT, Qwen, LLaMA) sono tutti "figli" dell'architettura Transformer.

## Prerequisiti e Setup
Installiamo le librerie necessarie (Transformers, PyTorch, Pandas, Scikit-learn).

In [ ]:
!pip install -q transformers datasets torch pandas scikit-learn

import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, confusion_matrix

# **La pipeline di Hugging Face: La scatola magica**
Hugging Face è l'azienda che ha creato la libreria open-source transformers, diventata lo standard mondiale per l'IA testuale. Al suo interno, hanno inserito uno strumento chiamato pipeline.

La pipeline è un'astrazione di alto livello: una funzione che raggruppa in una sola riga di codice tutte le operazioni complesse che altrimenti ti richiederebbero decine di righe di codice matematico in PyTorch o TensorFlow.

Quando scrivi pipeline("sentiment-analysis", model="...") e le passi un testo, lei fa automaticamente tre cose al posto tuo:

 - Pre-elaborazione (Tokenizer): Prende il tuo testo grezzo e lo trasforma in ID numerici e tensori che la rete neurale può digerire (aggiungendo i token speciali come [CLS], facendo il padding, ecc.).

 - Inferenza (Modello): Passa i numeri dentro le centinaia di strati della rete neurale Transformer (i pesi) e ottiene i logits (i punteggi grezzi in uscita).

 - Post-elaborazione (Output): Prende i punteggi matematici astratti e applica delle funzioni (come la Softmax) per trasformarli in probabilità leggibili da un umano (es. Label: 5 stars, Score: 89%).

In [ ]:
# 1. Importiamo lo strumento pipeline dalla libreria
from transformers import pipeline

# 2. Inizializziamo la pipeline per l'analisi del sentiment.
# Usiamo un modello basato su BERT addestrato a capire l'italiano e assegnare "stelle" (da 1 a 5).
# La prima volta che esegui questo codice su Colab, scaricherà automaticamente i pesi del modello.
analizzatore = pipeline(
    task="text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

# 3. Creiamo una lista di frasi di test
recensioni = [
    "Questo corso è fantastico, sto imparando un sacco di cose nuove!",
    "Pessima esperienza, il codice era pieno di bug e non funzionava niente.",
    "Non male, alcune parti sono un po' noiose ma nel complesso è utile.",
    "Che capolavoro, un'opera davvero incommensurabile!"
]

# 4. Passiamo l'intera lista alla pipeline
risultati = analizzatore(recensioni)

# 5. Stampiamo i risultati estratti dal dizionario
print("--- RISULTATI DELL'ANALISI ---\n")
for frase, risultato in zip(recensioni, risultati):
    etichetta = risultato['label']
    sicurezza = risultato['score'] * 100

    print(f"Testo: '{frase}'")
    print(f"Previsione: {etichetta} (Sicurezza: {sicurezza:.1f}%)\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--- RISULTATI DELL'ANALISI ---

Testo: 'Questo corso è fantastico, sto imparando un sacco di cose nuove!'
Previsione: 5 stars (Sicurezza: 84.1%)

Testo: 'Pessima esperienza, il codice era pieno di bug e non funzionava niente.'
Previsione: 1 star (Sicurezza: 92.9%)

Testo: 'Non male, alcune parti sono un po' noiose ma nel complesso è utile.'
Previsione: 3 stars (Sicurezza: 73.6%)

Testo: 'Che capolavoro, un'opera davvero incommensurabile!'
Previsione: 5 stars (Sicurezza: 95.3%)



## Blocco 1: Anatomia del Tokenizer e Tensori


### Cos'è BERT?

**BERT** (acronimo di *Bidirectional Encoder Representations from Transformers*) è un modello linguistico sviluppato da Google.

Per capire cos'è in termini semplici, immagina che nel mondo dell'intelligenza artificiale esistano principalmente due tipi di "studenti":

1. **I Generatori (Decoder):** Come ChatGPT, Qwen o LLaMA. Il loro compito è indovinare la parola successiva. Leggono da sinistra verso destra (es: "Il gatto dorme sul..." -> *tappeto*).
2. **I Lettori / Analizzatori (Encoder):** Questo è **BERT**. BERT non è fatto per generare testo nuovo o chiacchierare. Il suo superpotere è **comprendere a fondo il significato del testo**.

**Come fa a capire così bene? (La Bidirezionalità)**
La genialità di BERT è la sua "bidirezionalità". Invece di leggere la frase una parola dopo l'altra da sinistra a destra, BERT guarda *l'intera frase tutta insieme*, studiando sia le parole che vengono prima di un termine sia quelle che vengono dopo (il contesto di destra e di sinistra).

Se tu scrivi la parola "banca", BERT capisce immediatamente dal contesto della frase se stai parlando di una "banca" per prelevare soldi o di una "banca" del sangue, perché osserva l'intera stringa simultaneamente.

**A cosa serve in pratica?**
Viene usato nei motori di ricerca o nelle aziende per tutti i task dove è necessario *analizzare* e non inventare:

* **Sentiment Analysis:** Capire se una recensione è positiva o negativa.
* **Named Entity Recognition (NER):** Estrarre automaticamente nomi, indirizzi o importi in denaro da un'email.
* **Question Answering:** Trovare l'esatto paragrafo che risponde a una domanda all'interno di un lungo documento legale (estraendo il testo, non riscrivendolo a parole sue).
* **Classificazione testuale:** Smistare le mail in arrivo in cartelle ("Assistenza", "Fatturazione", "Spam") in base al contenuto.

BERT usa il WordPiece Tokenizer. Aggiunge token speciali (`[CLS]` e `[SEP]`) e richiede una maschera di attenzione (`attention_mask`) per il padding.

Il **WordPiece Tokenizer** è lo specifico algoritmo di segmentazione del testo (tokenizzazione) inventato da Google e utilizzato da modelli come **BERT**.

Il suo scopo è trovare il compromesso perfetto tra il dividere una frase in "parole intere" (che creerebbe un dizionario infinito) e dividerla in "singole lettere" (che farebbe perdere il significato originario e allungherebbe troppo i calcoli). Lo fa utilizzando i **sottotermini (subwords)**.

Ecco come funziona nel dettaglio:

### 1. La logica del "Pezzo più lungo possibile"

Quando WordPiece legge una parola, prima cerca di vedere se esiste intera nel suo vocabolario pre-addestrato (che per BERT è di sole 30.522 parole/sottoparole).

* Se la parola esiste (es. "casa"), la lascia intatta: `[casa]`
* Se la parola **non esiste** (perché è rara, composta, o contiene un errore di battitura), non va in crash. Inizia a scomporla nei "pezzi" più lunghi che conosce, partendo dall'inizio della parola.

### 2. Il prefisso magico: `##`

Quando WordPiece spezza una parola in più frammenti, deve ricordarsi che quei frammenti originariamente formavano un'unica parola, in modo da non confonderli con parole separate da spazi.
Per farlo, aggiunge il simbolo `##` davanti a tutti i frammenti successivi al primo. Il `##` significa letteralmente: *"attaccami al token precedente senza metterci uno spazio in mezzo"*.

**Esempio pratico:**
Immagina di far leggere a BERT la parola rara *"psiconeuroimmunologia"*. Questa parola intera non esiste nel suo vocabolario. WordPiece la taglierà così:
`["psico", "##neuro", "##immun", "##ologia"]`

Se invece gli passi un neologismo o un nome inventato come *"Goku"*, potrebbe dividerlo in:
`["go", "##ku"]`

### I 3 Vantaggi principali di WordPiece

* **Mai più Out-Of-Vocabulary (OOV):** I vecchi sistemi andavano in panico quando incontravano una parola mai vista (restituendo un token di errore come `<UNK>`). WordPiece non va mai in errore: nel peggiore dei casi assoluti, scomporrà una parola aliena fino alle sue singole lettere (poiché l'alfabeto di base è sempre incluso nel vocabolario).
* **Comprensione morfologica:** Dividendo le parole in radici e suffissi (es. `["gioca", "##tore"]`, `["gioca", "##trice"]`), il modello impara la grammatica da solo. Capisce che la radice "gioca" porta il significato base, mentre i suffissi "tore/trice" indicano chi compie l'azione.
* **Efficienza di memoria:** Permette a BERT di capire quasi ogni lingua del mondo memorizzando una tabella di sole 30.000 voci, un numero ridicolmente piccolo rispetto ai milioni di parole esistenti.

### Differenza con gli altri modelli (BPE)

Mentre BERT usa **WordPiece**, i modelli generativi come GPT e LLaMA usano algoritmi chiamati **BPE (Byte-Pair Encoding)**. Il risultato pratico è quasi identico (entrambi usano le sottoparole), ma cambia la matematica con cui scelgono quali lettere fondere insieme durante l'addestramento del vocabolario: BPE unisce le lettere che *appaiono più spesso vicine*, WordPiece unisce le lettere che *massimizzano la probabilità statistica complessiva* del testo.

## Token speciali di BERT

Quando diamo in pasto un testo a modelli come BERT, il tokenizer non si limita a spezzare le parole in frammenti (WordPiece), ma aggiunge dei "segnali stradali" (i token speciali) e fornisce una mappa per la lettura (la maschera di attenzione).

Questi elementi sono indispensabili per tradurre il linguaggio umano nella matematica rigida dei tensori (matrici) richiesta dalla rete neurale.

### I Token Speciali: `[CLS]` e `[SEP]`

Poiché BERT è un modello che non legge il testo per generare una risposta discorsiva, ma lo legge per **analizzarlo**, ha bisogno di punti di riferimento espliciti all'interno della frase.

* **`[CLS]` (Classification):** Viene inserito **sempre all'inizio** di ogni input. Il suo compito è fare da "riassunto" dell'intera frase. Durante l'addestramento, BERT viene forzato a comprimere il significato globale di tutto il testo successivo dentro questo specifico token. Se vuoi chiedere al modello "questa email è spam o no?", non analizzi l'output di ogni singola parola dell'email, ma prendi l'output (l'embedding) generato dal token `[CLS]` e lo usi per la decisione finale.
* **`[SEP]` (Separator):** Viene inserito **alla fine** di una frase. Serve a dire al modello "qui finisce il pensiero". Diventa ancora più cruciale quando chiedi a BERT di confrontare due frasi diverse (ad esempio in un task di *Question Answering* in cui passi un testo e una domanda). In quel caso, i token saranno disposti così: `[CLS] Testo del paragrafo [SEP] Qual è la domanda? [SEP]`. In questo modo BERT sa esattamente dove finisce il contesto e dove inizia la domanda.

### Il Padding e la Maschera di Attenzione (`attention_mask`)

Le reti neurali funzionano tramite moltiplicazioni di matrici, e le matrici richiedono dimensioni fisse. Non puoi passare a BERT prima una matrice di 3 parole e subito dopo una di 15 parole. Se elabori i dati in "batch" (più frasi contemporaneamente per velocizzare i calcoli), tutte le frasi devono avere la stessa identica lunghezza.

* **Il Padding (`[PAD]`):** Per risolvere il problema delle lunghezze diverse, si prende la frase più lunga del gruppo e si allungano artificialmente tutte le altre aggiungendo un token vuoto chiamato `[PAD]` (che corrisponde all'ID 0) finché non raggiungono la dimensione massima.
* **La Maschera di Attenzione (`attention_mask`):** Se aggiungiamo dei token `[PAD]` finti per pareggiare le lunghezze, rischiamo che il "meccanismo di attenzione" di BERT (la sua capacità di capire quali parole sono legate tra loro) impazzisca cercando di trovare un significato in questi zeri fittizi, rovinando l'analisi. La *maschera di attenzione* risolve il problema. È un array composto da **1** e **0** fornito insieme al testo:
* **1** indica un token reale (incluse le parole, `[CLS]` e `[SEP]`). Il modello ci "presterà attenzione".
* **0** indica un token `[PAD]`. Dice al modello: "Ignora completamente questo spazio, esiste solo per fare volume matematico".



**Esempio Visivo:**
Se stiamo elaborando due frasi e la lunghezza massima è fissata a 7 token:

**Frase 1:** "Adoro studiare"

* **Token:** `[CLS]` `Adoro` `studiare` `[SEP]` `[PAD]` `[PAD]` `[PAD]`
* **ID:** `[101, 15309, 2910, 102, 0, 0, 0]`
* **Attention Mask:** `[1, 1, 1, 1, 0, 0, 0]`

**Frase 2:** "Oggi piove molto forte"

* **Token:** `[CLS]` `Oggi` `piove` `molto` `forte` `[SEP]` `[PAD]`
* **ID:** `[101, 3502, 8312, 1920, 2030, 102, 0]`
* **Attention Mask:** `[1, 1, 1, 1, 1, 1, 0]`

In [ ]:
nome_modello = "bert-base-multilingual-uncased"
tokenizer = AutoTokenizer.from_pretrained(nome_modello)

testo = "L'intelligenza artificiale è incommensurabilmente affascinante."

tokens = tokenizer.tokenize(testo)
print("1. Token Grezzi (WordPiece):", tokens)

ids = tokenizer.convert_tokens_to_ids(tokens)
print("2. ID del vocabolario:", ids)

inputs = tokenizer(testo, return_tensors="pt", padding=True, truncation=True)
print("\n3. Tensori finali per PyTorch:")
print("Input IDs:", inputs["input_ids"])
print("Attention Mask:", inputs["attention_mask"])


1. Token Grezzi (WordPiece): ['l', "'", 'intel', '##ligen', '##za', 'artificial', '##e', 'e', 'inc', '##ommen', '##sura', '##bilmente', 'af', '##fas', '##cina', '##nte', '.']
2. ID del vocabolario: [154, 112, 23086, 27252, 10601, 30621, 10111, 147, 13565, 55667, 95006, 78167, 10513, 42681, 20038, 11223, 119]

3. Tensori finali per PyTorch:
Input IDs: tensor([[  101,   154,   112, 23086, 27252, 10601, 30621, 10111,   147, 13565,
         55667, 95006, 78167, 10513, 42681, 20038, 11223,   119,   102]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


### Esercizio (Ora 1)
Scrivi una funzione che accetti una lista di frasi di lunghezze diverse e stampi i tensori con il padding corretto. Osserva come l'Attention Mask posiziona gli `0`.

In [ ]:
# Scrivi qui il tuo codice per l'esercizio dell'Ora 1
def padding_correct(lista_frasi):
    inputs = tokenizer(lista_frasi, return_tensors="pt", padding=True, truncation=True)

    print("Input IDs (con padding al massimo batch):")
    print(inputs["input_ids"])
    print("\nAttention Mask:")
    print(inputs["attention_mask"])

frasi = ["L'acqua non è verde", "Make it as simple as possible, but not simpler.", "Che torta deliziosa 🏠⋆˚.♡˚⋆🏠"]
padding_correct(frasi)

Input IDs (con padding al massimo batch):
tensor([[  101,   154,   112, 23583, 10466,   147, 16716,   102,     0,     0,
             0,     0,     0,     0],
        [  101, 12696, 10197, 10146, 15511, 10146, 14312,   117, 10502, 10497,
         15511, 10131,   119,   102],
        [  101, 10283, 88186, 10112, 27254, 93780, 10112,   100,   119,   100,
           102,     0,     0,     0]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])


## Blocco 2: Sotto il cofano - Il Modello Base e gli Embeddings
Invece di usare AutoModelForSequenceClassification (che ha già un classificatore in cima), useremo AutoModel. Questo ci restituisce il vero output di BERT: gli Hidden States (vettori densi a 768 dimensioni che rappresentano il significato contestuale della frase).

In [ ]:
# Carichiamo il modello BASE (senza la "testa" di classificazione)
modello_base = AutoModel.from_pretrained(nome_modello)

# Disattiviamo il calcolo dei gradienti (fondamentale per non saturare la RAM durante l'inferenza)
with torch.no_grad():
    output = modello_base(**inputs)

# L'output principale è last_hidden_state (le feature finali)
hidden_states = output.last_hidden_state

print("Forma del tensore:", hidden_states.shape)
# La forma sarà [Batch_size, Sequence_length, 768]
# Es: [1, 14, 768] significa 1 frase, 14 token, ogni token è un vettore di 768 numeri.

# IL TOKEN CLS:
# Il primissimo token di ogni frase per BERT è sempre [CLS] (ID 101).
# Questo token è stato addestrato per aggregare il significato dell'INTERA frase.
cls_embedding = hidden_states[:, 0, :]
print("Forma dell'embedding [CLS]:", cls_embedding.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Forma del tensore: torch.Size([3, 14, 768])
Forma dell'embedding [CLS]: torch.Size([3, 768])


### Esercizio (Ora 2)
Calcola il tensore medio (Mean Pooling) di tutti i token della frase, escludendo il padding (usando l'Attention Mask), e confrontalo con il token `[CLS]`.

In [ ]:
import torch
from torch.nn.functional import cosine_similarity

# Assumiamo di avere già 'hidden_states' (l'output di BERT) e 'inputs' (che contiene l'attention_mask)

# 1. Estrarre il token [CLS] (posizione 0 di ogni frase nel batch)
cls_embedding = hidden_states[:, 0, :]

# 2. Calcolare il Mean Pooling (Media dei token veri)
# La nostra attention_mask ha forma [batch_size, seq_len] (es. [1, 14]).
# Dobbiamo espanderla a [batch_size, seq_len, 768] per poterla moltiplicare con gli hidden_states.
mask_espansa = inputs["attention_mask"].unsqueeze(-1).expand(hidden_states.size()).float()

# Azzera i vettori dei token di padding (quelli con mask = 0 diventeranno array di zeri)
vettori_filtrati = hidden_states * mask_espansa

# Somma i vettori di tutti i token per ogni frase (lungo la dimensione della sequenza: dim=1)
somma_vettori = torch.sum(vettori_filtrati, dim=1)

# Conta quanti token reali ci sono per ogni frase per fare la media (somma degli 1 nella mask)
# Usiamo clamp(min=1e-9) per sicurezza, per evitare eventuali divisioni per zero
conteggio_token_veri = torch.clamp(mask_espansa.sum(dim=1), min=1e-9)

# Calcola la media finale dividendo la somma per il numero di token reali
mean_pooling_embedding = somma_vettori / conteggio_token_veri

# 3. Stampiamo i risultati e confrontiamoli
print("Forma dell'embedding [CLS]:", cls_embedding.shape)
print("Forma dell'embedding Mean Pooling:", mean_pooling_embedding.shape)

# Calcoliamo la Similarità del Coseno tra i due vettori per ogni frase nel batch
similarita = cosine_similarity(cls_embedding, mean_pooling_embedding)

print("\n--- Similarità per singola frase ---")
for i, frase in enumerate(frasi):
    print(f"Frase {i+1}: \"{frase}\"")
    print(f"-> Similarità tra [CLS] e Mean Pooling: {similarita[i].item():.4f}\n")

Forma dell'embedding [CLS]: torch.Size([3, 768])
Forma dell'embedding Mean Pooling: torch.Size([3, 768])

--- Similarità per singola frase ---
Frase 1: "L'acqua non è verde"
-> Similarità tra [CLS] e Mean Pooling: 0.3658

Frase 2: "Make it as simple as possible, but not simpler."
-> Similarità tra [CLS] e Mean Pooling: 0.1115

Frase 3: "Che torta deliziosa 🏠⋆˚.♡˚⋆🏠"
-> Similarità tra [CLS] e Mean Pooling: 0.2343



Quale rappresenta meglio la frase?

La risposta breve è: Dipende dal task, ma in genere per estrarre feature "crude" il Mean Pooling è migliore.

 - Il token [CLS] (L'approccio ufficiale di classificazione):
Durante l'addestramento originale di BERT, il modello è stato forzato a usare esclusivamente il token [CLS] per indovinare se due frasi erano consecutive o no (Next Sentence Prediction). Per questo motivo, se stai aggiungendo un livello per classificare un testo (es. Sentiment Analysis o rilevamento Spam) e hai intenzione di addestrare (fine-tunare) quel livello, il token [CLS] è perfetto. Tuttavia, se lo estrai "a crudo" dal modello base (senza fine-tuning), tende ad essere un po' troppo focalizzato sugli aspetti strutturali del pre-addestramento e non cattura in modo ottimale il significato semantico puro.

 - Il Mean Pooling (L'approccio per la semantica):
Facendo la media di tutti i token (escludendo il padding), forzi l'embedding finale a tenere conto del "peso" semantico di ogni singola parola della frase. Numerosi studi hanno dimostrato che il Mean Pooling produce vettori (embeddings) migliori per valutare la somiglianza semantica tra due frasi a crudo. Ad esempio, architetture rivoluzionarie per i motori di ricerca semantici, come Sentence-BERT (SBERT), sono nate proprio sostituendo l'output del token [CLS] con il Mean Pooling per generare "riassunti" vettoriali più accurati delle frasi.

## Blocco 3: Integrazione con workflow di Data Mining
Ora che sappiamo come estrarre il vettore [CLS] (che è una "feature" perfetta del testo), possiamo usare BERT come un semplice estrattore di caratteristiche e passare questi vettori a un classico modello di Machine Learning (es. Regressione Logistica o Random Forest)

In [ ]:
from sklearn.linear_model import LogisticRegression

dataset = pd.DataFrame({
    "testo": [
        "Affronterò qualsiasi pericolo per proteggere i miei amici.",
        "Il potere e l'ambizione sono le uniche cose che contano.",
        "Lo studio e la logica sono le armi più affilate.",
        "La lealtà e il duro lavoro ripagano sempre."
    ],
    "etichetta": [1, 0, 2, 3]
})

def estrai_features(testi):
    inputs = tokenizer(testi.tolist(), padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = modello_base(**inputs)
    return out.last_hidden_state[:, 0, :].numpy()

X = estrai_features(dataset["testo"])
y = dataset["etichetta"]

print("Dimensioni Matrice X:", X.shape)
print("Pronto per l'addestramento scikit-learn!")


Dimensioni Matrice X: (4, 768)
Pronto per l'addestramento scikit-learn!


## Blocco 4: Il Progetto Pratico e le Metriche di Valutazione
Scarica o crea un dataset di 200-300 frasi testuali, estrai gli embedding, dividi i dati e addestra un classificatore classico (SVM, LogisticRegression). Misurane le performance.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Definisci X e y (usa il tuo dataset reale qui)
# 2. Dividi il dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Addestra e Valuta
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
predizioni = clf.predict(X_test)

print("--- Report di Classificazione ---")
print(classification_report(y_test, predizioni))
print("Matrice di Confusione:\n", confusion_matrix(y_test, predizioni))
